# 05. Agent 통합 테스트

Day 2의 마지막 단계. 04에서 단독 검증된 도구 3개(guide·weather·currency)를 `create_react_agent`로 묶고, plan Section 11 시연 시나리오 5개를 실제 Agent로 돌려본다.

**검증 포인트**
1. Agent가 질문 종류에 따라 적절한 도구를 자동 선택하는가
2. 03_rag_test에서 잡지 못한 시점·수치 환각이 도구 통합 후 자연 해소되는가 (Day 1 인계 항목)
3. 페르소나 가드가 작동해 사이판 정보가 답변에 섞이지 않는가 (plan 위험 신호 표)
4. MemorySaver + thread_id로 후속 질문에서 이전 맥락을 기억하는가 (시연 #5)
5. 환율 환산 계산이 LLM 쪽에서 정확히 이루어지는가 (옵션 A 분리 검증)

**시연 시나리오 5개** (plan Section 11)
1. 순수 RAG: "PIC 리조트 어때? 가족이 가도 좋아?"
2. 도구 1개 — 날씨: "5월 둘째 주 괌 날씨 어때? 우비 챙겨야 해?"
3. 도구 1개 — 환율: "100달러면 한국 돈 얼마야?"
4. 멀티 도구 + RAG: "PIC 추천해줘. 4박 5일 30만원이면 충분해?"
5. 메모리: 시연 #4 후속 — "그럼 더 저렴한 곳은?"

## 1. 환경 설정 + Agent 객체 import

In [1]:
import sys
from pathlib import Path

# 노트북 작업 디렉토리는 practice/, 그 한 단계 위 src/를 path에 추가
sys.path.insert(0, str(Path.cwd().parent / "src"))

from langchain_core.messages import AIMessage, ToolMessage

from guam_chatbot.agent import agent, invoke_agent, TOOL_NAMES

print(f"Agent 타입: {type(agent).__name__}")
print(f"Agent 노드: {list(agent.nodes.keys())}")
print(f"\n등록된 도구:")
for name in TOOL_NAMES:
    print(f"  - {name}")

c:\Users\wonwo\fastcampus\09.LLM\project\guam-family-chatbot\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Agent 타입: CompiledStateGraph
Agent 노드: ['__start__', 'agent', 'tools']

등록된 도구:
  - search_guam_guide
  - get_guam_weather
  - convert_currency


## 헬퍼 — 시연 결과 출력 포맷

각 시연마다 응답 시간, 도구 호출 흐름, 최종 답변을 일관되게 보여주도록 헬퍼 정의.

In [2]:
import time

def run_demo(question: str, thread_id: str, label: str, verbose: bool = False):
    """시연 시나리오 1건 실행 + 결과 정리 출력.
    
    verbose=True면 ToolMessage 본문(LLM이 실제로 받은 raw 도구 결과)까지 출력.
    환각 진단용: AIMessage 답변에 있는 정보가 ToolMessage에 정말 있는지 직접 확인.
    """
    print("=" * 70)
    print(f"[{label}] thread_id={thread_id}")
    print(f"Q: {question}")
    print("=" * 70)

    start = time.time()
    result = invoke_agent(question, thread_id=thread_id)
    elapsed = time.time() - start

    msgs = result["messages"]
    tool_calls = [m for m in msgs if isinstance(m, ToolMessage)]

    print(f"\n⏱️  응답 시간: {elapsed:.2f}초")
    print(f"📞 도구 호출 횟수: {len(tool_calls)}")
    for tm in tool_calls:
        print(f"   - {tm.name}")

    if verbose:
        print(f"\n=== 메시지 흐름 (환각 진단용) ===")
        for i, m in enumerate(msgs):
            kind = type(m).__name__
            if isinstance(m, ToolMessage):
                print(f"\n--- [{i}] {kind} (tool={m.name}) ---")
                print(m.content)  # raw 도구 결과 — 길이 제한 없음
            elif isinstance(m, AIMessage):
                if m.tool_calls:
                    print(f"\n--- [{i}] {kind} (도구 호출 결정) ---")
                    for tc in m.tool_calls:
                        print(f"  → {tc['name']}({tc.get('args', {})})")
                else:
                    print(f"\n--- [{i}] {kind} (최종 답변) ---")
                    print(m.content)
            else:
                print(f"\n--- [{i}] {kind} ---")
                print(m.content[:300])
    else:
        print(f"\n=== 최종 답변 ===")
        print(msgs[-1].content)
    print()
    return result

## 2. 시연 #1 — 순수 RAG

Q: "PIC 리조트 어때? 가족이 가도 좋아?"

**기대 동작**: search_guam_guide 1회 호출 → PIC 정보 기반 답변 + 출처 인용

**확인할 것**
- search_guam_guide만 호출되는지 (날씨·환율 도구 안 부르는지)
- 답변에 "PIC"·"가족"·"한국어 직원"·"수영장" 등 핵심 정보 포함
- 출처 인용 (예: `[리조트] 퍼시픽 아일랜드 클럽 - 나무위키`)
- ⚠️ **사이판 PIC 정보가 답변에 섞이지 않는지** (페르소나 가드 검증)

In [3]:
_ = run_demo(
    question="PIC 리조트 어때? 가족이 가도 좋아?",
    thread_id="demo1",
    label="시연 #1 — 순수 RAG",
)

[시연 #1 — 순수 RAG] thread_id=demo1
Q: PIC 리조트 어때? 가족이 가도 좋아?

⏱️  응답 시간: 3.34초
📞 도구 호출 횟수: 1
   - search_guam_guide

=== 최종 답변 ===
PIC 리조트는 가족 여행에 정말 좋아요! 나무위키에 따르면, 괌 최대 규모의 수영장과 어린이 체험 프로그램이 마련되어 있어 아이들이 신나게 놀 수 있답니다. 특히 워터파크, 키즈 클럽, 무료 액티비티(양궁, 패들보트 등)가 있어 하루 종일 즐길 거리가 가득해요.  

뷔페도 한식·중식·일식·양식 다양하게 제공되니 식사 걱정도 덜 수 있고, 한국인 직원이 상주해 언어 장벽 없이 편리하게 이용 가능합니다. 다만 시설이 오래된 점은 감안해야 하지만, 가족 친화적인 서비스로 충분히 커버된다고 해요.  

추가로, PIC는 투몬만 근처에 위치해 해변 접근성도 좋으니 가족 여행에 최적이에요! 😊  

[리조트] 퍼시픽 아일랜드 클럽 (PIC) - 나무위키



## 3. 시연 #2 — 도구 1개 (날씨)

Q: "5월 둘째 주 괌 날씨 어때? 우비 챙겨야 해?"

**기대 동작**: get_guam_weather 1회 호출 → 5/8~5/13 일별 데이터 기반 답변

**확인할 것**
- get_guam_weather만 호출되는지
- 03_rag_test의 시점·수치 환각("70~100mm" 등)이 사라졌는지 → **Day 1 인계 검증 포인트**
- 비 확률 100% 일자가 답변에 반영되어 "우비 챙기세요" 식 결론 도출
- weather.py의 어색한 한국어 description("실 비"/"온흐림")이 LLM이 자연스럽게 풀어주는지

In [4]:
_ = run_demo(
    question="5월 둘째 주 괌 날씨 어때? 우비 챙겨야 해?",
    thread_id="demo2",
    label="시연 #2 — 날씨",
)

[시연 #2 — 날씨] thread_id=demo2
Q: 5월 둘째 주 괌 날씨 어때? 우비 챙겨야 해?

⏱️  응답 시간: 3.16초
📞 도구 호출 횟수: 1
   - get_guam_weather

=== 최종 답변 ===
5월 둘째 주 괌 날씨는 평균 27°C로 덥지만, **5월 9일(토)부터 13일(수)까지 비 확률이 69%~100%로 매우 높아요!** 특히 10일(일)~13일(수)은 거의 매일 비가 예상되니 **우비와 우산은 필수**로 챙기셔야 합니다.  

- **5월 8일(금)**: 비 확률 20% (우비 없어도 OK)  
- **5월 9일(토)~13일(수)**: 비 확률 69%~100% (우비·우산 필수)  

실내 액티비티나 비 오는 날 갈 만한 장소를 미리 계획해 두시면 좋을 것 같아요! ☔



## 4. 시연 #3 — 도구 1개 (환율)

Q: "100달러면 한국 돈 얼마야?"

**기대 동작**: convert_currency 1회 호출 → 환율값 받아서 LLM이 100×환율 계산해 답변

**확인할 것**
- convert_currency만 호출되는지
- 옵션 A 분리 검증: 도구는 환율만, 환산(`100 × 1450.80 = 145,080원`)은 LLM이 직접
- 답변에 환산 계산 과정 또는 결과 포함 (대략 14만~15만원 수준)
- 답변에 영업일·매매기준율 안내 자연스럽게 녹여내는지

In [5]:
_ = run_demo(
    question="100달러면 한국 돈 얼마야?",
    thread_id="demo3",
    label="시연 #3 — 환율",
)

[시연 #3 — 환율] thread_id=demo3
Q: 100달러면 한국 돈 얼마야?

⏱️  응답 시간: 2.25초
📞 도구 호출 횟수: 1
   - convert_currency

=== 최종 답변 ===
100달러 × 1,450.80원 = **145,080원**입니다.  
(2026년 5월 8일 한국은행 기준)  

*최신 환율로 정확히 환산해 드렸어요!* 😊



## 5. 시연 #4 — 멀티 도구 (RAG + 환율)

Q: "PIC 추천해줘. 4박 5일 30만원이면 충분해?"

**기대 동작**: search_guam_guide + convert_currency 둘 다 호출 → PIC 정보 + 환율 환산 통합 답변

**확인할 것**
- 도구 2개 모두 호출되는지 (Agent의 도구 선택 로직 검증)
- 30만원 ≈ ~$200 환산이 답변에 들어가는지
- PIC 가격 정보가 RAG에 없으므로 "가격은 자료에서 확인되지 않습니다" 솔직 답변
  → **시연 시나리오의 함정**: 30만원이 충분한지에 대한 단정적 답변을 피하고 한계 인정
- thread_id `demo4` — 시연 #5에서 이어서 사용

In [6]:
demo4_result = run_demo(
    question="PIC 추천해줘. 4박 5일 30만원이면 충분해?",
    thread_id="demo4",
    label="시연 #4 — 멀티 도구 (RAG + 환율)",
    verbose=True,  # ← 추가
)

[시연 #4 — 멀티 도구 (RAG + 환율)] thread_id=demo4
Q: PIC 추천해줘. 4박 5일 30만원이면 충분해?

⏱️  응답 시간: 4.23초
📞 도구 호출 횟수: 2
   - search_guam_guide
   - convert_currency

=== 메시지 흐름 (환각 진단용) ===

--- [0] HumanMessage ---
PIC 추천해줘. 4박 5일 30만원이면 충분해?

--- [1] AIMessage (도구 호출 결정) ---
  → search_guam_guide({'query': '가족 여행자에게 추천하는 괌 리조트 PIC'})
  → convert_currency({'query': '100달러'})

--- [2] ToolMessage (tool=search_guam_guide) ---
[리조트] 퍼시픽 아일랜드 클럽 (PIC) - 나무위키
1. 개요

Pacific Islands Club, PIC

괌 및 북마리아나 제도에 위치한 미국의 복합형 리조트. 흔히 PIC라는 약칭으로 불린다. 괌 및 사이판에 여행하는 관광객들이 가장 많이 찾는 숙박시설이다.

한국인, 일본인 관광객들이 가장 많이 숙박하며, 각 리조트에는 한국인, 일본인 직원 및 현지인 중 한국어 및 일본어 구사가 가능한 직원이 배치되어있다.

워낙 한국인 관광객이 많이 찾다 보니, 한국어 사이트도 개설되어 있다.[5]
2. PIC 괌
괌 투몬만에 위치한 퍼시픽 아일랜드 클럽 괌
PIC 하면 생각나는 리조트이며, PIC 리조트 중 가장 큰 리조트이다. 괌에서 최대 규모를 자랑하며, 괌을 찾는 관광객들이 가장 많이 애용한다. 특히 가족 단위 관광객들에게 특화되어 있으며 다양한 레크리에이션과, 괌 최대 규모의 수영장, 그리고 어린이 관광객을 위한 체험 프로그램이 마련되어 있다.

투숙하는 기간 내내 매일 석식뷔페에 간다면 알 수 있겠지만 오늘 봤던 한국인 손님은 다음날엔 없고 전날에 못봤던 다른 한국인 손님으로 채워지는 등 매일매일 회전하듯이 바뀐다.[6]

## 6. 시연 #5 — 메모리 (시연 #4 후속)

Q: "그럼 더 저렴한 곳은?"

**기대 동작**: thread_id `demo4`로 다시 호출 → MemorySaver가 #4 맥락 자동 로드 → "PIC 외 더 저렴한 옵션"으로 해석

**확인할 것**
- thread_id `demo4`가 #4와 동일 — 메모리 연결
- "그럼"이 무엇을 가리키는지 LLM이 #4 맥락에서 판단 (PIC 비교 대상)
- search_guam_guide 다시 호출 (다른 리조트 정보 찾기)
- 답변이 PIC 외 다른 리조트를 언급하는지 (RAG에 다른 리조트 정보 있다면)

In [7]:
_ = run_demo(
    question="그럼 더 저렴한 곳은?",
    thread_id="demo4",
    label="시연 #5 — 메모리 (#4 후속)",
    verbose=True,  # ← 추가
)

[시연 #5 — 메모리 (#4 후속)] thread_id=demo4
Q: 그럼 더 저렴한 곳은?

⏱️  응답 시간: 6.54초
📞 도구 호출 횟수: 4
   - search_guam_guide
   - convert_currency
   - search_guam_guide
   - search_guam_guide

=== 메시지 흐름 (환각 진단용) ===

--- [0] HumanMessage ---
PIC 추천해줘. 4박 5일 30만원이면 충분해?

--- [1] AIMessage (도구 호출 결정) ---
  → search_guam_guide({'query': '가족 여행자에게 추천하는 괌 리조트 PIC'})
  → convert_currency({'query': '100달러'})

--- [2] ToolMessage (tool=search_guam_guide) ---
[리조트] 퍼시픽 아일랜드 클럽 (PIC) - 나무위키
1. 개요

Pacific Islands Club, PIC

괌 및 북마리아나 제도에 위치한 미국의 복합형 리조트. 흔히 PIC라는 약칭으로 불린다. 괌 및 사이판에 여행하는 관광객들이 가장 많이 찾는 숙박시설이다.

한국인, 일본인 관광객들이 가장 많이 숙박하며, 각 리조트에는 한국인, 일본인 직원 및 현지인 중 한국어 및 일본어 구사가 가능한 직원이 배치되어있다.

워낙 한국인 관광객이 많이 찾다 보니, 한국어 사이트도 개설되어 있다.[5]
2. PIC 괌
괌 투몬만에 위치한 퍼시픽 아일랜드 클럽 괌
PIC 하면 생각나는 리조트이며, PIC 리조트 중 가장 큰 리조트이다. 괌에서 최대 규모를 자랑하며, 괌을 찾는 관광객들이 가장 많이 애용한다. 특히 가족 단위 관광객들에게 특화되어 있으며 다양한 레크리에이션과, 괌 최대 규모의 수영장, 그리고 어린이 관광객을 위한 체험 프로그램이 마련되어 있다.

투숙하는 기간 내내 매일 석식뷔페에 간다면 알 수 있겠지만 오늘 봤던 한국인 손님은 다음날엔 없고 전날에 못봤던 다른 한국인 손님으로

## 7. 검증 결과 ✅

| 시연 | 응답 시간 | 도구 호출 | 핵심 검증 | 결과 |
|---|---|---|---|---|
| #1 순수 RAG | ?초 | ? | PIC 정보 + 출처, 사이판 미혼입 | ? |
| #2 날씨 | ?초 | ? | 시점·수치 환각 해소, 우비 결론 | ? |
| #3 환율 | ?초 | ? | 옵션 A 분리, LLM이 환산 계산 | ? |
| #4 멀티 | ?초 | ? | RAG + 환율 둘 다, 가격 한계 인정 | ? |
| #5 메모리 | ?초 | ? | #4 맥락에서 "그럼" 해석 | ? |

**Day 2 끝 체크 (plan Section 9)**
- ✅/❌ "5월 괌 날씨 어때?" 동작
- ✅/❌ "PIC 리조트 추천" 동작
- ✅/❌ "30만원이면 며칠 놀아?" 동작

**알려진 이슈 / 개선 포인트** (Day 3 발표 폴리싱 후보)
- (실행 후 발견되는 것 기록)

**다음 단계**: schemas.py 정의 (보강 B) + Day 2 GitHub 커밋

In [ ]:
# ===== 환각 빈도 측정 (시연 #4 5회 회귀) =====
# 환각 정의: 답변에 자료에 없는 가격 견적/호텔명 등장
# 매 호출 새 thread_id → 이전 호출 메모리 영향 배제

import re
import uuid

QUERY = "5인 가족이 5월 둘째 주에 괌 갈 건데, PIC 어때? 날씨도 보고 싶고 100달러 환전 환율도 알려줘"
N = 5

# 환각 패턴 (Day 2~3 측정 결과 기반)
HALLUCINATION_PATTERNS = [
    (r'\d+\s*~\s*\d+\s*만\s*원',     '가격 견적 범위 (X~Y만 원)'),
    (r'약\s*\d+\s*만\s*원',          '약 ~만원 우회 표현'),
    (r'평균\s*\d+\s*(만\s*원|달러)', '평균 ~만원/달러 우회 표현'),
    (r'추정\s*\d+',                  '추정 ~ 우회 표현'),
    (r'대략\s*\d+',                  '대략 ~ 우회 표현'),
    (r'하몬\s*인|Harmon\s*Inn',      '환각 호텔명: 하몬 인'),
    (r'로열\s*라구나|Royal\s*Laguna','환각 호텔명: 로열 라구나'),
    (r'호텔스닷컴|아고다|익스피디아', 'Day 2 환각 사이트명'),
]

results = []
for i in range(N):
    print(f"\n{'='*60}\n호출 {i+1}/{N}\n{'='*60}")
    
    # 새 thread_id (이전 호출 메모리 영향 배제)
    thread_id = f"measure-{uuid.uuid4().hex[:8]}"
    
    result = invoke_agent(QUERY, thread_id=thread_id)
    answer = result["messages"][-1].content
    
    # 자동 환각 검출 (보조 — 최종 판정은 사용자 눈으로)
    detected = []
    for pattern, label in HALLUCINATION_PATTERNS:
        matches = re.findall(pattern, answer)
        if matches:
            detected.append((label, matches))
    
    is_halluc = len(detected) > 0
    results.append({"call": i+1, "answer": answer, "auto_halluc": is_halluc, "detected": detected})
    
    print(f"\n[답변]\n{answer}\n")
    print(f"[자동 검출] {'⚠️ 환각 의심' if is_halluc else '✓ 깨끗 (자동 패턴 기준)'}")
    for label, matches in detected:
        print(f"  - {label}: {matches}")

# 집계
print(f"\n{'='*60}")
print(f"자동 검출 기준 환각 빈도: {sum(r['auto_halluc'] for r in results)}/{N}")
print(f"{'='*60}")
print("\n⚠️ 자동 검출은 보조. 최종 판정은 위 5개 답변 직접 읽고 판단하세요:")
print("  - 가격 견적이 자료에 없는데 등장하는가?")
print("  - 호텔명이 자료에 없는데 등장하는가?")
print("  - '약/평균/추정/대략' 같은 우회 표현이 등장하는가?")


호출 1/5

[답변]
**PIC 리조트 추천**  
[리조트] 퍼시픽 아일랜드 클럽 (PIC) - 나무위키에 따르면, PIC는 가족 여행에 최적화된 리조트입니다.  
- **장점**:  
  - 한국인 직원 상주, 한국어 서비스 가능  
  - 대형 수영장, 키즈 클럽, 무료 액티비티(양궁, 워터 슬라이드 등)  
  - 매일 다양한 뷔페(한식·중식·일식·양식) 제공  
- **주의**: 시설이 오래되었으나 서비스로 커버 가능  
- **추천 이유**: 5인 가족(특히 어린이 동반)에게 적합  

**5월 둘째 주 날씨**  
- 5/8(금): 26~27°C, 비 확률 20% (우비 준비 권장)  
- 5/9(토)~5/13(수): 26~28°C, **비 확률 69~100%** (우천 대비 필수)  
- **팁**: 실내 액티비티(키즈 클럽, 수영장) 활용 추천  

**100달러 환율**  
- 2026년 5월 8일 기준: **1 USD = 1,450.80 KRW**  
- **환산**: 100달러 × 1,450.80 = **145,080원**  

*PIC 예약 시 성수기(5월) 가격 변동 가능성 있으니 공식 사이트 확인 필수!*

[자동 검출] ✓ 깨끗 (자동 패턴 기준)

호출 2/5

[답변]
**PIC 리조트 추천**  
PIC는 괌 최대 규모의 가족 친화형 리조트로, 어린이 체험 프로그램과 대형 수영장이 있어 5인 가족에게 적합합니다. 한식·중식·일식 뷔페도 제공되며, 한국인 직원이 상주해 언어 불편이 적습니다. (출처: 나무위키)  

**5월 둘째 주 날씨**  
- 5/9(토): 비 확률 69% (평균 27°C)  
- 5/10(일)~5/13(수): 비 확률 100% (평균 27°C)  
→ 우비와 우산을 꼭 준비하세요!  

**100달러 환전 금액**  
- 100달러 × 1,450.80원 = **145,080원** (2026-05-08 기준)  

※ PIC 시설 노후화는 있지만, 워터파크와 액티비티로 커버 가능합니다. 비 예보가 많으

In [10]:
# ===== 시연 #4 + #5 메모리 흐름 1회 동작 확인 =====
# 같은 thread_id로 #4 → #5 연속 호출. #5의 "그럼"이 #4 맥락(PIC, 5월 둘째 주, 100달러)을 잇는지.

thread_id = "demo45"

print("=" * 60)
print("[시연 #4 (재설계)]")
print("=" * 60)
q4 = "5인 가족이 5월 둘째 주에 괌 갈 건데, PIC 어때? 날씨도 보고 싶고 100달러 환전 환율도 알려줘"
r4 = invoke_agent(q4, thread_id=thread_id)
print(r4["messages"][-1].content)

print("\n" + "=" * 60)
print("[시연 #5 (메모리)]")
print("=" * 60)
q5 = "그럼 다른 추천도 있어?"
r5 = invoke_agent(q5, thread_id=thread_id)
print(r5["messages"][-1].content)

[시연 #4 (재설계)]
**PIC 리조트 추천**  
- PIC는 괌 최대 규모 리조트로 가족 여행에 최적화되어 있어요.  
- 한국인 직원 상주, 한국어 사이트 제공, 다양한 레크리에이션(워터파크, 어린이 체험 프로그램)이 장점입니다.  
- 단, 시설이 오래됐다는 점은 참고 부탁드려요! [출처: 나무위키]  

**5월 둘째 주 날씨**  
- 5/9(토)~5/13(수) 대부분 비 예상 (강수확률 69~100%).  
- 평균 기온 26~28°C로 무더위보다는 선선할 예정이에요.  

**100달러 환전 금액**  
- 2026-05-08 기준 환율: 1 USD = 1,450.80 KRW  
- 100달러 × 1,450.80 = **145,080원**  

※ 비 예보가 많으니 우비나 우산을 꼭 챙기시고, PIC 예약 시 공식 사이트에서 최신 가격 확인해주세요!

[시연 #5 (메모리)]
**PIC 외 가족 추천 리조트**  
1. **투몬만 지역 호텔**  
   - 괌 대표 관광지 투몬만에 위치한 고층 호텔들이 많아요.  
   - 와이키키 해변과 유사한 분위기로 가족 단위 관광객에게 인기입니다.  
   - [출처: Wikivoyage]  

2. **하몬 지역 호텔**  
   - 공항 근처 하몬 지역에 저렴한 숙소가 많지만, 번화가와 거리가 있어 조용한 분위기를 선호하는 가족에게 적합합니다.  
   - 단, 하몬은 산업/주거 지역이 혼재되어 있어 시설 상태가 다소 떨어질 수 있다는 점은 참고해주세요.  

**비 오는 날 추천 활동**  
- PIC 내 **실내 워터파크**나 **어린이 체험 프로그램**을 활용해보세요.  
- 투몬만 호텔의 **실내 수영장**이나 **뷔페 식당**에서 편안하게 시간을 보내는 것도 좋습니다.  

※ PIC는 비 오는 날에도 즐길 수 있는 시설이 잘 갖춰져 있어, 5월 둘째 주처럼 비가 잦은 시기에 특히 추천해요!
